# **Retrieval-Augmented Generation (RAG) System for Domain-Specific Question Answering**

**Problem Statement:** Develop a simple Retrieval-Augmented Generation (RAG) system to answer questions from custom documents. Build a pipeline that retrieves relevant information from a document and uses a language model to generate answers.

## **Executive Summary**
This notebook implements an end-to-end Retrieval-Augmented Generation (RAG) pipeline designed to extract grounded, context-aware answers from custom unstructured data. The architecture utilizes a hybrid retrieval optimization strategy (combining dense vector semantics with sparse keyword frequencies) and integrates with the Google Gemini API for advanced reasoning and factual generation.

---
## **Instruction Guide: How to Run This Project in Google Colab**
To evaluate this pipeline securely using the Gemini API, follow these steps to configure your environment:
1. Open this notebook in Google Colab.
2. Locate the **"Secrets"** tab on the left-hand sidebar (represented by a **🔑 key icon**).
3. Click **"Add new secret"**.
4. In the *Name* field, enter exactly: `GEMINI_API_KEY`
5. In the *Value* field, paste your personal Gemini API key (generated from Google AI Studio).
6. **Critical:** Toggle the switch next to the secret to grant **"Notebook access"**.
7. Click `Runtime > Run all` from the top menu to execute the pipeline.


## **Section 1: Environment Configuration and Dependency Installation**
Installs all necessary parsers, vector engines, and machine learning libraries.

In [1]:
!pip install -q pypdf sentence-transformers faiss-cpu rank_bm25 datasets google-genai
print("System Log: Core dependencies installed successfully (including modern google-genai SDK)")

System Log: Core dependencies installed successfully (including modern google-genai SDK)


## **Section 2: Universal Data Ingestion and Chunking Methodology**
This module standardizes unstructured inputs. To mitigate context fragmentation, the chunking algorithm applies a strict token-limit with a defined sequential overlap, ensuring semantic boundaries are preserved.

In [2]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import re
import numpy as np
import faiss
from pypdf import PdfReader
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

class UniversalIngestionModule:
    @staticmethod
    def load_pdf(file_path):
        reader = PdfReader(file_path)
        return "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])

    @staticmethod
    def load_txt(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    @staticmethod
    def load_huggingface(dataset_name, text_column, split_config="train[:20]"):
        print(f"System Log: Initiating stream from Hugging Face repository -> {dataset_name}")
        ds = load_dataset(dataset_name, split=split_config)
        return "\n\n".join([str(row[text_column]) for row in ds if row[text_column]])

    @staticmethod
    def chunk_text(raw_text, chunk_size=150, chunk_overlap=30):
        cleaned_text = re.sub(r'\s+', ' ', raw_text).strip()
        words = cleaned_text.split(' ')
        chunks = []
        start_idx = 0

        while start_idx < len(words):
            end_idx = min(start_idx + chunk_size, len(words))
            chunks.append(" ".join(words[start_idx:end_idx]))
            start_idx += (chunk_size - chunk_overlap)

        return [c for c in chunks if len(c.strip()) > 15]

print("System Log: Universal Ingestion Module initialized.")

System Log: Universal Ingestion Module initialized.


## **Section 3: Hybrid Search Engine Architecture**
**System Optimization Implementation:** The pipeline constructs a dual-index architecture. It processes queries through a dense semantic index (via FAISS) and a sparse keyword index (BM25Okapi), fusing the candidate sets prior to generation.

In [3]:
class HybridVectorEngine:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.embedding_model_name = model_name
        self.embedding_model = SentenceTransformer(model_name)
        self.embedding_dim = 384
        self.chunks = []
        self.faiss_index = None
        self.bm25 = None

    def build_index(self, text_chunks):
        self.chunks = text_chunks

        # Dense Semantic Mapping
        embeddings = self.embedding_model.encode(text_chunks, show_progress_bar=False)
        self.faiss_index = faiss.IndexFlatL2(self.embedding_dim)
        self.faiss_index.add(np.array(embeddings).astype('float32'))

        # Sparse Keyword Mapping
        self.bm25 = BM25Okapi([chunk.lower().split(" ") for chunk in text_chunks])
        print(f"System Log: Hybrid Vector Database constructed with {len(text_chunks)} chunk matrices.")

    def hybrid_retrieve(self, query, top_k=3):
        # Semantic Retrieval
        q_vec = self.embedding_model.encode([query]).astype('float32')
        _, dense_idx = self.faiss_index.search(q_vec, top_k * 2)

        # Keyword Retrieval
        sparse_scores = self.bm25.get_scores(query.lower().split(" "))
        sparse_idx = np.argsort(sparse_scores)[::-1][:top_k * 2]

        # Index Fusion
        candidates = list(set(list(dense_idx[0]) + list(sparse_idx)))
        valid_candidates = [i for i in candidates if i != -1 and i < len(self.chunks)]

        return [self.chunks[i] for i in valid_candidates][:top_k]

print("System Log: Hybrid Vector Engine initialized.")

System Log: Hybrid Vector Engine initialized.


## **Section 4: Grounded Language Model Pipeline (API Integration)**
Utilizes the Gemini API for advanced contextual reasoning. The prompt is strictly constrained to prevent hallucination and isolate answers solely to the retrieved context.

In [4]:
from google import genai
from google.genai import types
from google.colab import userdata

class APIBasedGenerationModule:
    def __init__(self):
        try:
            # Securely fetch the API key
            GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
            # The SDK uses a Client architecture
            self.client = genai.Client(api_key=GOOGLE_API_KEY)
        except userdata.SecretNotFoundError:
            raise ValueError("CRITICAL ERROR: GEMINI_API_KEY not found in Colab Secrets.")

        # Targeting the active, current model
        self.model_id = 'gemini-2.5-flash'

    def generate_answer(self, query, contexts):
        combined_context = "\n\n".join([f"[Context Block {i+1}]: {text}" for i, text in enumerate(contexts)])

        # System instructions are  passed cleanly via the config object
        sys_instruct = "You are a precise analytical assistant. Answer the user's question using ONLY the provided context blocks. If the context does not contain the answer, state that explicitly. Do not invent details."

        prompt = f"Context:\n{combined_context}\n\nQuestion: {query}\n\nAnswer:"

        # Generating content
        response = self.client.models.generate_content(
            model=self.model_id,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=sys_instruct,
                temperature=0.1
            )
        )

        return response.text

print("System Log: API-Based Generation Pipeline initialized (v2 SDK).")

System Log: API-Based Generation Pipeline initialized (v2 SDK).


## **Section 5: Pipeline Execution and Evaluation Dashboard**
**Instructions:** Adjust parameters below. Running this cell will trigger data ingestion, execution, and metric reporting seamlessly.

In [5]:
#@title  RAG System Evaluation Dashboard
Input_Source = "PDF Document" #@param ["PDF Document", "Raw Text File", "Hugging Face Dataset"]
User_Query = "what is he name of the person? what his 5 main strengths in bullet points? And what is main projects are and what is their importance?" #@param {type:"string"}
Chunk_Size = 150 #@param {type:"slider", min:50, max:500, step:50}
Chunk_Overlap = 30 #@param {type:"slider", min:10, max:100, step:10}
Retrieval_Count = 2 #@param {type:"slider", min:1, max:5, step:1}

from google.colab import files

print("-" * 80)
print(f"System Initialization: Mode Selected -> {Input_Source}")
print("-" * 80)

raw_corpus = ""

try:
    if Input_Source == "PDF Document":
        print("Awaiting PDF Upload via Colab Interface...")
        uploaded = files.upload()
        if not uploaded: raise ValueError("Upload interrupted.")
        file_name = list(uploaded.keys())[0]
        raw_corpus = UniversalIngestionModule.load_pdf(file_name)

    elif Input_Source == "Raw Text File":
        print("Awaiting TXT Upload via Colab Interface...")
        uploaded = files.upload()
        if not uploaded: raise ValueError("Upload interrupted.")
        file_name = list(uploaded.keys())[0]
        raw_corpus = UniversalIngestionModule.load_txt(file_name)

    elif Input_Source == "Hugging Face Dataset":
        hf_name = input("Enter Hugging Face Dataset identifier (e.g., 'squad'): ")
        hf_col = input("Enter the target text column name (e.g., 'context'): ")
        raw_corpus = UniversalIngestionModule.load_huggingface(hf_name, hf_col)

    if not raw_corpus.strip(): raise ValueError("Extracted data corpus is empty.")

    print("\nProcessing chunk boundaries and constructing Vector Database...")
    chunks = UniversalIngestionModule.chunk_text(raw_corpus, Chunk_Size, Chunk_Overlap)
    engine = HybridVectorEngine()
    engine.build_index(chunks)

    if 'generator' not in locals():
        generator = APIBasedGenerationModule()

    matched_contexts = engine.hybrid_retrieve(User_Query, top_k=Retrieval_Count)

    print("\n" + "="*80)
    print("                      VALIDATION LOGS: TEXT EXTRACTION                  ")
    print("="*80)
    for idx, block in enumerate(matched_contexts):
        print(f"[EXTRACTED CHUNK {idx+1}]:\n{block}\n")

    final_answer = generator.generate_answer(User_Query, matched_contexts)

    print("="*80)
    print("                     PIPELINE OUTPUT: GROUNDED RESPONSE                 ")
    print("="*80)
    print(f"{final_answer}\n")

    print("="*80)
    print("                            SYSTEM METRICS REPORT                       ")
    print("="*80)
    print(f" * Data Ingestion Source         : {Input_Source}")
    print(f" * Token Window Segment Size     : {Chunk_Size} tokens")
    print(f" * Total Segment Matrices Mapped : {len(chunks)} blocks")
    print(f" * Embedding Model               : {engine.embedding_model_name}")
    print(f" * Text Embedding Dimensions     : {engine.embedding_dim}")
    print(f" * Semantic Vector Engine        : FAISS (IndexFlatL2) + Sparse BM25")
    print(f" * Generative Environment        : Google Gemini 2.5 Flash (google-genai SDK)")
    print("="*80)

except Exception as e:
    print(f"\nCRITICAL PIPELINE ERROR: {str(e)}")
    print("Please verify your inputs and API keys.")

--------------------------------------------------------------------------------
System Initialization: Mode Selected -> PDF Document
--------------------------------------------------------------------------------
Awaiting PDF Upload via Colab Interface...


Saving Nimendra_Giri_Resume.pdf to Nimendra_Giri_Resume (6).pdf

Processing chunk boundaries and constructing Vector Database...


System Log: Hybrid Vector Database constructed with 4 chunk matrices.

                      VALIDATION LOGS: TEXT EXTRACTION                  
[EXTRACTED CHUNK 1]:
Nimendra Giri Data Science | Data Analyst Enthusiast ☎ +91 7891841680 | ✉ nimendragiri@gmail.com | 🔗 /in/nimendra-giri | 💻 Github/Nimendra_Giri 📍 Jaipur, Rajasthan Career Objective MCA student with hands-on experience in data science using Python, Pandas, Scikit-learn, and Seaborn, with strong interest in data science, machine learning, AI, and data analysis. Passionate about solving real-world problems and building scalable, user-focused solutions while expanding skills in Azure ML and cloud technologies. Educational Qualification Master of Computer Applications | JECRC University, Jaipur | CGPA: 9.32 2025 – 2027 Bachelor of Computer Applications | Seth G.L. Bihani S.D. PG College, MGSU | Percentage: 86 2022 – 2025 Skills Technical Skills • Languages: Python (Intermediate), Java, C++, SQL, HTML • Libraries: Pandas, NumPy, 

## **Project Metrics Overview**
| Architecture Component | Implementation Metric |
| :--- | :--- |
| **Text Embedding Dimensions** | `384` Dimensions |
| **Base Chunk Window** | `150` Tokens |
| **Chunk Overlap Size** | `30` Tokens |
| **Optimization Strategy** | Hybrid Search (Dense FAISS + Sparse BM25) |
| **Generative LLM Setup** | Google Gemini 2.5 Flash via `google-genai` SDK |

---
## **Conclusion**
This Retrieval-Augmented Generation (RAG) project successfully demonstrates the integration of robust document ingestion, semantic chunking, and advanced hybrid retrieval strategies. By coupling dense semantic vectors with sparse keyword tracking, the pipeline addresses retrieval limitations often seen in highly technical datasets. Furthermore, leveraging the Google Gemini 2.5 Flash API ensures the system generates highly accurate, contextually bounded responses anchored strictly to the ingested domain data. The entire pipeline is driven through an interactive parameterized dashboard, allowing evaluators to dynamically adjust datasets and inspect retrieval behavior. Ultimately, this architecture serves as a scalable blueprint for deploying secure, offline-ready AI assistance over proprietary knowledge bases, mitigating the risk of language model hallucination.
